<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/projects/04-trace-the-team/notebook.ipynb)


# Project 04 — Trace the team, then grade it

Project 03 told you how often the team picked the right company. It could not tell you
why it was ever wrong. Today it can.


## The brief

The desk acts on the team's answers now, so somebody eventually asks the question you
cannot answer yet: **what went wrong on that one?**

Project 03 keeps a receipt. `calls` says a model was asked three times. `stopped_because`
says the run finished. `rejected` says a citation was dropped. Every one of those is a
total, and none of them says in what order anything happened. So none of them separates
the two failures that look identical from outside:

- the passage never reached the prompt, and
- the passage reached the prompt and the model ignored it.

The first is a retrieval fix. The second is a prompt fix. Guess wrong and you spend a week
in the wrong file.

**You will not edit the team.** You rarely own the code you need to trace. What you do own
are the seams you inject into it: the team takes a `search` and a model, so you wrap those
two and the run records itself.


## What you deliver

| You deliver | Step | What it is | Check |
|---|---|---|---|
| `trace` | 1 | the ordered events of one run, each a `kind` and a `detail` | `project-04-e1` |
| `cases` | 2 | the labelled questions as a golden set | `project-04-e2` |
| `report` | 3 | one outcome per case: passed or not, and the reason | `project-04-e3` |
| `buckets` | 4 | how many failures fell in each of the five buckets | `project-04-e4` |

Four different shapes on purpose: a list of events, a list of cases, a list of outcomes, a
dict of counts. Project 03 shipped with two checks wanting the same shape, so handing one
the other's deliverable still went green. These refuse each other by name.


## The data

Nothing new. You read what Projects 02 and 03 already produced.

| Path | What it is |
|---|---|
| `../02-sec-filings/data/raw/*.html` | Item 1A of eight companies' Form 10-K, read by path |
| `../02-sec-filings/data/questions.json` | 20 questions, each labelled with the company that answers it |
| `../03-analyst-team/data/recorded/recorded.json` | one real run's model replies, replayed offline |


## Before you start

```bash
uv sync --extra projects --extra agents
uv run jupyter lab
```

Ollama is optional and this notebook does not use it. **You grade on the recorded lane on
purpose.** A live 7B model words the same answer two ways on two runs, so a pass rate
measured live moves for reasons that are not fixes.


## How to use this notebook

Run it top to bottom. Four steps, each ending in a check you can read. The checks pin no
answer and count toward no mark: they say whether the shape you built is the shape the
next step needs.

Read every trace before you name a bucket. That order is the whole project.


In [ ]:
# manual-run: reads Project 02's filings from disk, so CI does not execute it.
# Setup: find the course, load the checks, and read what Projects 02 and 03 left behind.
import json
import math
import re
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from bootcamp_agent.agent import FAILURE_BUCKETS, TRACE_KINDS
from bootcamp_agent.bonus import BONUS
from bootcamp_agent.bonus import bonus as check_step
from bootcamp_agent.llm import FakeLLM
from bootcamp_agent.projects import sec_filings
from bootcamp_agent.projects.analyst_team import build_team, companies
from bootcamp_agent.projects.trace_and_grade import labelled_questions
from bootcamp_agent.retrieval import _tokens as tokens

RECORDED_FILE = ROOT / "projects" / "03-analyst-team" / "data" / "recorded" / "recorded.json"
RECORDED = json.loads(RECORDED_FILE.read_text(encoding="utf-8"))
REPLIES = RECORDED.get("replies", {})
PROVENANCE = RECORDED.get("_provenance", {})

print(f"answers:  [recorded] {PROVENANCE.get('recorded', '?')} from {PROVENANCE.get('model', '?')}")
print(f"replies:  {len(REPLIES)} keyed on the text of the question")
print(f"kinds:    {list(TRACE_KINDS)}")
print(f"buckets:  {list(FAILURE_BUCKETS)}")


def verdict(check_id, value):
    """Ask what a check WOULD say about something, without handing it in.

    The Try it cells use this. `check_step` records your deliverable; this only
    reads. Keeping them apart is why an experiment cannot overwrite your answer.
    """
    print(BONUS[check_id](value) or "no complaint")


In [ ]:
# The tool, rebuilt in one cell. This is Project 03's search, and it is not what you
# are here to learn: it exists so there is something real to trace.
def passages_of(ticker):
    """One company's filing as passages: tags gone, page furniture gone, short lines gone."""
    parser = sec_filings._Visible()
    parser.feed(sec_filings.raw_filing(ticker))
    out = []
    for line in "".join(parser.parts).split("\n"):
        line = re.sub(r"\s+", " ", line).strip()
        if not line or any(pattern.match(line) for pattern in sec_filings.NOISE) or len(line) < 80:
            continue
        out.append(line[:800])
    return out


INDEX = [
    (f"{entry['ticker'].lower()}#{position}", text)
    for entry in sec_filings.sources()
    for position, text in enumerate(passages_of(entry["ticker"].lower()))
]
PASSAGE_WORDS = [set(tokens(text)) for _, text in INDEX]
DOCUMENT_FREQUENCY = Counter(word for words in PASSAGE_WORDS for word in words)


def search(question, ticker=None, k=4):
    """Shared words, each weighted by how rare it is. Nothing is written."""
    asked = set(tokens(question))
    scored = []
    for position, words in enumerate(PASSAGE_WORDS):
        chunk_id, text = INDEX[position]
        if ticker and not chunk_id.startswith(f"{ticker}#"):
            continue
        score = sum(math.log(1 + len(INDEX) / DOCUMENT_FREQUENCY[word]) for word in asked & words)
        if score > 0:
            scored.append((round(score, 2), chunk_id, text))
    return sorted(scored, key=lambda row: (-row[0], row[1]))[:k]


print(f"{len(INDEX)} passages over {len(sec_filings.sources())} companies")


## 1. Trace a run you did not write

The team is somebody else's code, and you are not going to open it.

Look at what `build_team` takes: a `search` and a model. Those two arguments are the seams.
Everything the team learns about the world comes through `search`, and every word it
produces comes through the model. Wrap both and the run records itself, in order, with the
team untouched.

An event needs two things to be worth keeping: a **kind**, so you can count them, and a
**detail**, so you can tell one from another. The four kinds are declared in
`bootcamp_agent.agent` rather than invented here, because a fifth kind spelled two ways is
a trace you cannot compare between runs.

**What to look at:** the order of the printed events. Retrieval happens once and the model
is called four times. A receipt would have told you `calls` had four entries in it; only
the order tells you the passages were there before the first word was written.


In [ ]:
# The two wrappers. Neither one changes what the team does; both record that it happened.
def traced_search(inner, events):
    """Same contract as search, plus one 'retrieve' event per call."""
    def wrapper(question, ticker=None, k=4):
        rows = inner(question, ticker, k)
        events.append(
            {"kind": "retrieve", "detail": f"ticker={ticker} k={k} -> {[row[1] for row in rows]}"}
        )
        return rows

    return wrapper


class TracedModel:
    """Same contract as any LLMClient, plus one 'llm_call' event per call."""

    def __init__(self, inner, events):
        self.inner = inner
        self.events = events

    def complete(self, system, user):
        reply = self.inner.complete(system, user)
        role = system.strip().splitlines()[0][:40] if system.strip() else "(no system prompt)"
        self.events.append({"kind": "llm_call", "detail": f"{role} -> {len(reply)} chars"})
        return reply


def run_traced(question):
    """One run of the untouched team, and the ordered events it produced."""
    events = []
    team = build_team(
        traced_search(search, events),
        TracedModel(FakeLLM(responses=REPLIES), events),
        max_revisions=1,
        budget=6,
    )
    state = team.run(question)
    answer = state.get("answer")
    events.append(
        {
            "kind": "decision",
            "detail": f"stopped_because={state.get('stopped_because')} "
            f"citations={list(getattr(answer, 'citations', ()))}",
        }
    )
    return state, events


In [ ]:
# One run, and the trace it left. This is the deliverable for step 1.
cases_preview = labelled_questions()
first_state, trace = run_traced(cases_preview[0]["question"])

print(cases_preview[0]["question"])
print()
for event in trace:
    print(f"  [{event['kind']:9}] {event['detail'][:96]}")


In [ ]:
# Check step 1.
check_step("project-04-e1", trace)


In [ ]:
# Try it: drop one kind of event, then the other, and watch which loss the check notices.
without_decision = [event for event in trace if event["kind"] != "decision"]
without_model = [event for event in trace if event["kind"] != "llm_call"]

print("without the decision event:")
verdict("project-04-e1", without_decision)
print()
print("without the model calls:")
verdict("project-04-e1", without_model)
print()
print("One of those passed. A check can only refuse an absence it was told to look for,")
print("and every trace you will ever read is missing something nobody named.")


<details><summary>Hint 1</summary>

`build_team(search, model, ...)` takes both seams as arguments. You never have to open `analyst_team.py`.

</details>

<details><summary>Hint 2</summary>

A wrapper has to return exactly what the inner function returned, or the team behaves differently while being watched.

</details>

<details><summary>Hint 3</summary>

`TRACE_KINDS` is imported in the setup cell. If you want a fifth kind, the question to answer first is which page tells a reader where its fix lives.

</details>


> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain how step 1 of projects/04-trace-the-team/notebook.ipynb records a trace without editing the team. Do not change the code."
> - "What is the difference between the `calls` list Project 03 keeps and the trace built here?"
> - "Why does the wrapper return the inner result unchanged? What would break if it did not?"


## 2. The golden set from the labels

You are not going to invent twenty questions. Project 02 already labelled twenty, each with
the one company that answers it, and a label somebody checked is worth more than a label
you wrote while looking at the output.

**What to look at:** `labelled_questions()` reads those labels and nothing else. The check
refuses a company that appears in no source, because a golden set whose answers are not in
the corpus measures your typing.


In [ ]:
# The golden set. Read, not typed.
cases = labelled_questions()

print(f"{len(cases)} cases")
for case in cases[:3]:
    print(f"  {case['expected_company']:18} {case['question'][:64]}")


In [ ]:
# Check step 2.
check_step("project-04-e2", cases)


In [ ]:
# Try it: label one case with a company the corpus does not hold and run the check again.
broken = [dict(case) for case in cases]
broken[0]["expected_company"] = "Initech"
verdict("project-04-e2", broken)


<details><summary>Hint 1</summary>

`labelled_questions()` is imported in the setup cell. Print one row before you use twenty.

</details>

<details><summary>Hint 2</summary>

The check wants `question` and `expected_company` on every row. A row missing a label cannot be graded, so it is not a case.

</details>

<details><summary>Hint 3</summary>

Duplicates are refused because a repeated question counts one answer twice and quietly moves the rate.

</details>


> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain what labelled_questions() reads in projects/04-trace-the-team/notebook.ipynb and where those labels came from."
> - "Why does a golden set need labels somebody checked, rather than labels written after seeing the output?"
> - "What does project-04-e2 refuse, and why is each refusal there?"


## 3. Grade every case

A pass condition is a decision you write down. Ours grades **routing**: did the team send
the question to the right company?

That choice is forced by the lane, and it is worth saying out loud. The recording covers
Project 03's demo questions, not all twenty, so most of these replay a stand-in reply.
Grading the answer text offline would grade the stand-in. Routing is decided with **no
model call at all**, so it is real on the recorded lane and it is real on a live one.

**What to look at:** the pass rate, and then the three rows that failed. A report where
everything passed is refused by the check, and the message says why.


In [ ]:
# The pass condition, written down before the run.
NAMES = companies()


def grade(case):
    """Route the question, compare against the label, and keep the trace that decided it."""
    state, events = run_traced(case["question"])
    routed = NAMES.get(state.get("ticker") or "")
    passed = routed == case["expected_company"]
    return {
        "question": case["question"],
        "passed": passed,
        "reason": f"routed to {routed or 'nobody'}, expected {case['expected_company']}",
        "trace": events,
    }


graded = [grade(case) for case in cases]

# The deliverable is the report. The traces stay beside it, for step 4.
report = [{key: row[key] for key in ("question", "passed", "reason")} for row in graded]

hits = sum(row["passed"] for row in report)
print(f"pass rate: {hits}/{len(report)} = {hits / len(report):.0%}")
print()
for row in report:
    if not row["passed"]:
        print(f"  FAIL  {row['reason']}")
        print(f"        {row['question'][:78]}")


In [ ]:
# Check step 3.
check_step("project-04-e3", report)


In [ ]:
# Try it: mark every case passed and run the check again. The message is the point.
everything_green = [{**row, "passed": True} for row in report]
verdict("project-04-e3", everything_green)


<details><summary>Hint 1</summary>

`grade` returns the trace alongside the outcome. Step 4 needs it, and the deliverable for this step does not include it.

</details>

<details><summary>Hint 2</summary>

`passed` has to be a real boolean. A score between 0 and 1 is a thing you still have to make a decision about later.

</details>

<details><summary>Hint 3</summary>

If every case passes, the check refuses the report. Read one passing case by hand against its trace before you believe it.

</details>


> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain why the pass condition in step 3 of projects/04-trace-the-team/notebook.ipynb grades routing instead of the answer text."
> - "Why does project-04-e3 refuse a report where every case passed? Explain what it is protecting against."
> - "What does my pass condition not check? List the ways a wrong answer could still pass it."


## 4. Sort the failures into buckets

Now the traces earn their keep. A wrong answer looks the same whether retrieval missed or
the model ignored what it got. Only the ordered events tell those apart.

The five buckets are declared in `bootcamp_agent.agent`, and the bucket is the work item.
Four buckets holding one case each and one holding nine is not five problems, it is one.

**What to look at:** which bucket the failures land in, and then the count. Fix the biggest
bucket, not the first one you read.


In [ ]:
# Read each failure against its own trace, and name one bucket for it.
def bucket_of(row):
    """Did retrieval return anything for this run? The trace answers; the report cannot."""
    retrieved = [
        event
        for event in row["trace"]
        if event["kind"] == "retrieve" and not event["detail"].endswith("-> []")
    ]
    return "instruction_following" if retrieved else "retrieval"


buckets = dict(Counter(bucket_of(row) for row in graded if not row["passed"]))

print(f"{sum(buckets.values())} failures")
for name, count in sorted(buckets.items(), key=lambda pair: -pair[1]):
    print(f"  {name:22} {count}")


In [ ]:
# Check step 4.
check_step("project-04-e4", buckets)


In [ ]:
# Try it: invent a sixth bucket and run the check again. The refusal names the cost.
verdict("project-04-e4", {**buckets, "model_was_tired": 1})


<details><summary>Hint 1</summary>

`bucket_of` reads `row['trace']`, which `grade` kept for exactly this. The report alone cannot answer the question.

</details>

<details><summary>Hint 2</summary>

A retrieve event whose detail ends in `-> []` means nothing came back. That is the line between the two buckets.

</details>

<details><summary>Hint 3</summary>

`FAILURE_BUCKETS` is imported in the setup cell. A sixth bucket is a failure nobody will fix, because no page says where its fix lives.

</details>


> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Read one failing case in step 4 of projects/04-trace-the-team/notebook.ipynb with me and explain which bucket its trace points to. Do not tell me the bucket for the others."
> - "Why can the bucket only be decided from the trace and not from the answer text?"
> - "What would I lose if I dropped the retrieve events from the trace?"


## Measure

One number before, one number after, and a sentence saying what you changed. The point of
the buckets is that the second number is aimed.


In [ ]:
# Measure: the rate, the biggest bucket, and what a fix there would be worth.
biggest = max(buckets.items(), key=lambda pair: pair[1]) if buckets else ("none", 0)
ceiling = (hits + biggest[1]) / len(report)

print(f"{'pass rate now':24} {hits}/{len(report)} = {hits / len(report):.0%}")
print(f"{'biggest bucket':24} {biggest[0]} ({biggest[1]})")
print(f"{'if that bucket went to 0':24} {hits + biggest[1]}/{len(report)} = {ceiling:.0%}")
print()
print("That last line is a ceiling, not a result. It is what the fix could buy,")
print("and you only know what it did buy by rerunning this notebook after you make it.")


## Your turn

- **Widen the pass condition until a red case turns green**, then say out loud what you
  stopped checking. That is the trade, and naming it is the only thing that makes it
  honest. Widening after seeing the failure is fitting the test to the code.
- **Drop one event kind from the trace** and re-read a failure. Which two buckets can you
  no longer tell apart?
- **Shuffle the labels** so every `expected_company` is wrong, and regrade. Anything still
  passing is measuring your code rather than the team. This is the negative control from
  session 9, and it costs two lines.


## Resources

- Session 9, [trace and evaluate](../../units/en/unit2/session-09-trace-and-evaluate/introduction.mdx)
- Project 03, [the analyst team](../03-analyst-team/README.md)
- The checks you are running: `src/bootcamp_agent/projects/trace_and_grade.py`
- The kinds and the buckets, declared once: `src/bootcamp_agent/agent.py`


## Ask your assistant about this project

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain what projects/04-trace-the-team/notebook.ipynb builds, step by step, without changing any code."
> - "Why does this project wrap the team instead of editing analyst_team.py? Give me two reasons."
> - "Compare the receipt Project 03 keeps with the trace this project builds, field by field."
> - "I widened my pass condition and the rate went up. Help me say what I stopped checking."
> - "How would I add a tool_call event to this trace, and what would it let me tell apart?"
